# ACT 3: THE PEOPLE INSIDE THE SYSTEM 
## Hypothesis

Passenger tipping behavior changes during the stress zone (months 3–5),
indicating behavioral adaptation rather than purely mechanical system changes.

In [4]:
import sys
!"{sys.executable}" -m pip install scipy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ------ --------------------------------- 6.0/36.5 MB 45.5 MB/s eta 0:00:01
   ------------------------- -------------- 23.1/36.5 MB 65.3 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.5 MB 64.5 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 54.0 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

# Use same files
files = [
    "yellow_tripdata_2023-02.parquet",
    "yellow_tripdata_2023-03.parquet",
    "yellow_tripdata_2023-04.parquet",
    "yellow_tripdata_2023-05.parquet",
    "yellow_tripdata_2023-06.parquet"
]

cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "tip_amount"
]

df = pd.concat([
    pd.read_parquet(f, columns=cols).sample(frac=0.2)
    for f in files
], ignore_index=True)

# Feature engineering
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

df['trip_duration'] = (
    (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime'])
    .dt.total_seconds() / 60
)

# Clean data (Act 2 rules)
df = df[
    (df['trip_distance'] > 0) &
    (df['fare_amount'] > 0) &
    (df['trip_duration'] > 1) &
    (df['trip_duration'] < 120) &
    (df['tip_amount'] <= df['fare_amount'])
]

# Create features
df['tip_percentage'] = (df['tip_amount'] / df['fare_amount']) * 100
df['month'] = df['tpep_pickup_datetime'].dt.month

In [3]:
#SPLITTING DATA 
df_before = df[df['month'] < 3]
df_window = df[(df['month'] >= 3) & (df['month'] <= 5)]

In [4]:
#USING T-TEST - STATISTICAL TESTING 
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(
    df_before['tip_percentage'],
    df_window['tip_percentage'],
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 17.462096051948894
P-value: 2.857280096038317e-68


## Statistical Interpretation

If p-value < 0.05:
→ Significant difference → behavior changed

If p-value > 0.05:
→ No strong evidence of change

## Evaluation of Hypotheses

While statistical testing shows a difference in tipping behavior,
it is difficult to isolate purely behavioral causes.

The alternative explanation (pricing or trip composition changes)
is also plausible.

However, consistent increase in tip percentage suggests
a behavioral component, though not conclusively dominant.

## Final Conclusion

The analysis suggests that observed patterns in tipping behavior
are influenced by both behavioral and mechanical factors.

While statistical evidence indicates a shift during the stress period,
alternative explanations prevent a definitive conclusion.

This highlights the complexity of distinguishing human behavior
from system-level changes in real-world data.

Since the p-value is far less than 0.05, we reject the null hypothesis and conclude that there is a statistically significant difference in tipping behavior between the periods.

#hypothesis:
| Type       | Role                      |
| ---------- | ------------------------- |
| Behavioral | Main hypothesis          |
| Mechanical | Alternative   |


| Hypothesis         | Meaning                 |
| ------------------ | ----------------------- |
| **H₀ (Null)**      | Behavior did NOT change |
| **H₁ (Alternate)** | Behavior DID change     |
